In [ ]:
import sys

sys.path.append("/workspace/VLM2Vec")

from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import LoraConfig, get_peft_model, PeftModel
from src.model.model import MMEBModel
from src.arguments import ModelArguments
from src.model.processor import load_processor
from src.model.vlm_backbone.qwen2_vl.qwen_vl_utils import process_vision_info

MODEL = "/workspace/VLM2Vec/outputs/Qwen3.5-08b-merged-4750"
adapter_path = None
adapter_config = None


hf_config = AutoConfig.from_pretrained(MODEL, trust_remote_code=True)
if adapter_path:
    adapter_config = LoraConfig.from_pretrained(adapter_path, trust_remote_code=True)

/workspace/VLM2Vec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-05-21 10:58:48,161] INFO [src.model.vlm_backbone.qwen2_vl.qwen_vl_utils:41] set VIDEO_TOTAL_PIXELS: 90316800
[2026-05-21 10:58:48,170] DEBUG [httpcore.connection:47] connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
[2026-05-21 10:58:48,178] DEBUG [httpcore.connection:47] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7bc335b53ec0>
[2026-05-21 10:58:48,178] DEBUG [httpcore.connection:47] start_tls.started ssl_context=<ssl.SSLContext object at 0x7bc335b5d5d0> server_hostname='huggingface.co' timeout=10
[2026-05-21 10:58:48,182] DEBUG [httpcore.connection:47] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7bc33

In [2]:
import torch

model_args =ModelArguments(
    model_name=adapter_config.base_model_name_or_path if adapter_config else MODEL ,
    model_type=hf_config.model_type,
    processor_name=adapter_config.base_model_name_or_path if adapter_config else MODEL,
    model_backbone=hf_config.model_type,
    pooling="last",
    normalize=True,
    lora=True if adapter_config else False
)

processor = load_processor(model_args)
model = MMEBModel.load(model_args, is_trainable=False)
model = model.to('cuda', dtype=torch.bfloat16)
model.eval()

[2026-05-21 10:58:48,324] INFO [src.utils.basic_utils:21] Loading processor from: Qwen/Qwen3.5-0.8b
[2026-05-21 10:58:48,326] DEBUG [httpcore.http11:47] send_request_headers.started request=<Request [b'HEAD']>
[2026-05-21 10:58:48,326] DEBUG [httpcore.http11:47] send_request_headers.complete
[2026-05-21 10:58:48,327] DEBUG [httpcore.http11:47] send_request_body.started request=<Request [b'HEAD']>
[2026-05-21 10:58:48,327] DEBUG [httpcore.http11:47] send_request_body.complete
[2026-05-21 10:58:48,327] DEBUG [httpcore.http11:47] receive_response_headers.started request=<Request [b'HEAD']>
[2026-05-21 10:58:48,375] DEBUG [httpcore.http11:47] receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'88'), (b'Connection', b'keep-alive'), (b'Date', b'Thu, 21 May 2026 10:58:48 GMT'), (b'Location', b'/Qwen/Qwen3.5-0.8B/resolve/main/processor_config.json'), (b'X-Powered-By', b'huggingface-moon

MMEBModel(
  (encoder): Qwen3_5ForConditionalGeneration(
    (model): Qwen3_5Model(
      (visual): Qwen3_5VisionModel(
        (patch_embed): Qwen3_5VisionPatchEmbed(
          (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
        )
        (pos_embed): Embedding(2304, 768)
        (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
        (blocks): ModuleList(
          (0-11): 12 x Qwen3_5VisionBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): Qwen3_5VisionAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (mlp): Qwen3_5VisionMLP(
              (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
              (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
             

In [3]:
def encode_image(model, processor, image_path):
    inputs = processor(text=f'{VLM_IMAGE_TOKENS[QWEN3_5]} Represent the given image',
                   images=Image.open(image_path),
                   return_tensors="pt")

    inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
    inputs['image_grid_thw'] = inputs['image_grid_thw'].unsqueeze(0)
    inputs = {key: value.to('cuda') for key, value in inputs.items()}
    qry_output = model(qry=inputs)["qry_reps"]
    return qry_output

def encode_video(model, processor, video_path):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": "/workspace/VLM2Vec/assets/c2ed642b3a3a4430a2b1013e7a378b80_chunks_segment_002.mp4",
                    "max_pixels": 360 * 420,
                    "fps": 1.0,
                },
                {"type": "text", "text": "Describe this video."},
            ],
        }
    ]

    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=f'{VLM_VIDEO_TOKENS[QWEN3_5]} Represent the given video.',
        videos=video_inputs,
        return_tensors="pt"
    )

    inputs = {key: value.to('cuda') for key, value in inputs.items()}
    inputs['pixel_values_videos'] = inputs['pixel_values_videos'].unsqueeze(0)
    inputs['video_grid_thw'] = inputs['video_grid_thw'].unsqueeze(0)
    qry_output = model(qry=inputs)["qry_reps"]
    return qry_output

def encode_text(model, processor, text):
    inputs = processor(text=text,
                   images=None,
                   return_tensors="pt")
    inputs = {key: value.to('cuda') for key, value in inputs.items()}
    tgt_output = model(tgt=inputs)["tgt_reps"]
    return tgt_output


In [4]:
from src.model.processor import load_processor, QWEN3_5, VLM_IMAGE_TOKENS, VLM_VIDEO_TOKENS, Qwen3_5_process_fn
from PIL import Image

inputs = processor(text=f'{VLM_IMAGE_TOKENS[QWEN3_5]} Represent the given image',
                   images=Image.open('/workspace/VLM2Vec/assets/screenshot.png'),
                   return_tensors="pt")

inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
inputs['image_grid_thw'] = inputs['image_grid_thw'].unsqueeze(0)
inputs = {key: value.to('cuda') for key, value in inputs.items()}
qry_output = model(qry=inputs)["qry_reps"]

string = 'poem list in a table'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))

string = 'a poem about a cat'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))

string = 'two people walking on a beach'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))

[2026-05-21 10:58:58,829] DEBUG [PIL.Image:421] Importing PngImagePlugin
[2026-05-21 10:58:58,833] DEBUG [PIL.PngImagePlugin:204] STREAM b'IHDR' 16 13
[2026-05-21 10:58:58,833] DEBUG [PIL.PngImagePlugin:204] STREAM b'IDAT' 41 16384


poem list in a table = tensor([[0.5898]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)
a poem about a cat = tensor([[0.4062]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)
two people walking on a beach = tensor([[0.4980]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)


In [5]:
TEST_IMAGE = "/workspace/VLM2Vec/assets/example.jpg"

string = 'a cat and a dog'
qry_input = encode_image(model, processor, TEST_IMAGE)
tgt_output = encode_text(model, processor, string)
print(string, '=', model.compute_similarity(qry_input, tgt_output))

string = 'a table with list of issues'
tgt_output = encode_text(model, processor, string)
print(string, '=', model.compute_similarity(qry_input, tgt_output))


VIDEO_PATH = "/workspace/VLM2Vec/assets/c2ed642b3a3a4430a2b1013e7a378b80_chunks_segment_002.mp4"

qry_input = encode_video(model, processor, VIDEO_PATH)
string = 'a blue car on the accident on the road'
tgt_output = encode_text(model, processor, string)
print(string, '=', model.compute_similarity(qry_input, tgt_output))


string = 'a black car accident on the road'
tgt_output = encode_text(model, processor, string)
print(string, '=', model.compute_similarity(qry_input, tgt_output))

string = 'a red car accident on the road'
tgt_output = encode_text(model, processor, string)
print(string, '=', model.compute_similarity(qry_input, tgt_output))






[2026-05-21 10:59:17,778] DEBUG [PIL.Image:421] Importing JpegImagePlugin


a cat and a dog = tensor([[0.5039]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)


qwen-vl-utils using decord to read video.
[2026-05-21 10:59:18,727] INFO [src.model.vlm_backbone.qwen2_vl.qwen_vl_utils:248] decord:  video_path='/workspace/VLM2Vec/assets/c2ed642b3a3a4430a2b1013e7a378b80_chunks_segment_002.mp4', total_frames=206, video_fps=2.0, time=0.030s


a table with list of issues = tensor([[0.6367]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)


[transformers] Asked to sample `fps` frames per second but no video metadata was provided which is required when sampling with `fps`. Defaulting to `fps=24`. Please provide `video_metadata` for more accurate results.


a blue car on the accident on the road = tensor([[0.2012]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)
a black car accident on the road = tensor([[0.1641]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)
a red car accident on the road = tensor([[0.1602]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)


In [13]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "/workspace/VLM2Vec/assets/c2ed642b3a3a4430a2b1013e7a378b80_chunks_segment_002.mp4",
                "max_pixels": 360 * 420,
                "fps": 1.0,
            },
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=f'{VLM_VIDEO_TOKENS[QWEN3_5]} Represent the given video.',
    videos=video_inputs,
    return_tensors="pt"
)

inputs = {key: value.to('cuda') for key, value in inputs.items()}
inputs['pixel_values_videos'] = inputs['pixel_values_videos'].unsqueeze(0)
inputs['video_grid_thw'] = inputs['video_grid_thw'].unsqueeze(0)
qry_output = model(qry=inputs)["qry_reps"]

string = 'accident on the road'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))
## tensor([[0.4766]], device='cuda:0', dtype=torch.bfloat16)

string = 'A train running on a track.'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))
## tensor([[0.3262]], device='cuda:0', dtype=torch.bfloat16)

[2026-05-21 10:56:35,274] INFO [src.model.vlm_backbone.qwen2_vl.qwen_vl_utils:248] decord:  video_path='/workspace/VLM2Vec/assets/c2ed642b3a3a4430a2b1013e7a378b80_chunks_segment_002.mp4', total_frames=206, video_fps=2.0, time=0.032s


accident on the road = tensor([[0.2295]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)
A train running on a track. = tensor([[0.2578]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MmBackward0>)
